### This was the PCA Stuff with heatmap like figures

In [1]:
### Imports ------------------------------------------------------------------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler, random_split
import time
import random
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from typing import Callable, Sequence
import plotly.graph_objects as go
from tqdm import tqdm
import copy

from dash_stuff.design_test_dash import create_app, run_app

print(f"Cuda Check: {torch.cuda.is_available()}")

# load the data
n_samples_to_load = 5_000
with h5py.File("collections/2025-01-13--10-29-19--X10000.h5", "r") as f:
    data = [{key: np.array(f[item][key]) for key in f[item]} for item in list(f.keys())[:n_samples_to_load]]

Cuda Check: True


In [25]:
### Classes ------------------------------------------------------------------------------------------------------------
class MyDataset(Dataset):
    """
    Barebone dataset class
    """
    
    def __init__(self, data: Sequence):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        return self.data[idx] 

class ScheduledBatchSampler(Sampler):
    """
    batch_sampler with batch size scheduling based on a custom dict. Use as custom batch_sampler fn in Dataloader().
    batch schedule = {how many epochs : batch size, ... : ...}. the current epoch needs to be set in the training loop by using set_epoch(E). If more epochs are trained than specified in the schedule, the last batch size is returned
    """
    
    def __init__(self, data_source: Dataset, schedule: list[tuple], shuffle: bool=False, drop_last: bool=False):
        """
        Args:
            data_source: Dataset to sample from.
            schedule   : list of tuples with (BS, nr_epochs for that BS), e.g. [(32, 2), (64, 4), (128, 8)]
            shuffle    : whether to shuffle the dataset indices.
            drop_last  : whether to drop the last incomplete batch.
        """
        
        # set up instance variables
        self._data_source   = data_source                     # this is essentially the whole dataset
        self._num_samples   = len(data_source)                # length of the whole dataset
        self._indices       = list(range(self._num_samples))  # base indices. (shuffled) batches of these are returned
        self._schedule      = self._expand_schedule(schedule) # schedule list but now with each batch size repeated * n
        self._current_epoch = None                            # to keep track of the current epoch in training loop
        self._shuffle       = shuffle
        self._drop_last     = drop_last

    def __iter__(self):
        """
        this function is essentially responsible for returning a generator object. The yield statement is kind of like a return, but instead it just advances the loop further every time the next() method is called, or in a for loop. it kind of "remebers" the last time it was called.
        """
        
        if self._current_epoch is None:
            raise ValueError(f"no epoch set! always set current epoch of sampler in training loop with set_epoch!")
        
        # determine batch size
        batch_size = self._get_batch_size()
        
        # Shuffle before returning the generator object for the new epoch
        if self._shuffle:
            random.shuffle(self._indices) 

        # determines the start indices for each batch. range(start, stop, step). 
        # Note this is essentially only the index to find the indices in self.indices for one batch xD
        for start_idx in range(0, len(self._indices), batch_size):
            # grab all the indices (now those indices corresponding to samples) belonging to a batch
            batch = self._indices[start_idx:start_idx + batch_size]
            # skip yielding the last semi-full batch if self.drop_last is true
            if len(batch) < batch_size and self._drop_last:
                continue
            # basically returns one element when it"s called and then pauses the function execution until next()
            yield batch

    def __len__(self):
        """
        just returns the number of batches in the current epoch. changes by 1 depending on whether the last semi-full one is dropped or not.
        """
        
        if self._current_epoch is None:
            raise ValueError(f"no epoch set yet!")
        
        batch_size = self._get_batch_size()
        if self._drop_last:
            return math.floor(self._num_samples / batch_size)
        else:
            return math.ceil(self._num_samples / batch_size)

    def _expand_schedule(self, schedule: list[tuple]):
        """
        Expand the schedule to be a list of repeated batch size values. 
        e.g from [(64, 1), (128, 2), (256, 4)] -> [64, 128, 128, 256, 256, 256, 256]
        """
        
        expanded_schedule = []
        for batch_size, epochs in schedule:
            expanded_schedule.extend([batch_size] * epochs)
        return expanded_schedule

    def _get_batch_size(self):
        """
        just convenience for getting the currend batch size from the expanded schedule depending on current epoch. If the epoch is larger than what is specified in the schedule, it just returns the last value.
        """
        
        if self._current_epoch < len(self._schedule):
            return self._schedule[self._current_epoch]
        else:
            return self._schedule[-1]

    def get_total_epochs(self):
        """
        Returns the total number of epochs specified with the schedule
        """
        
        return len(self._schedule)
    
    def get_total_steps(self):
        """
        Returns the total number of batches that are returned by sampler. Takes drop_last into account.
        """
        
        total_steps = 0
        if self._drop_last is True:
            for bs in self._schedule:
                total_steps += math.floor(self._num_samples / bs)
        else:
            for bs in self._schedule:
                total_steps += math.ceil(self._num_samples / bs)
        return int(total_steps)
    
    def get_total_samples(self):
        """
        Returns the total number of samples that are returned by the sampler. Takes drop_last into account.
        """
        total_samples = 0
        
        if self._drop_last is True:
            for bs in self._schedule:
                total_samples += (self._num_samples // bs) * bs
        else:
            total_samples = len(self._schedule) * self._num_samples
        return int(total_samples)
    
    def set_epoch(self, epoch: int):
        """
        update the current epoch when called in the training loop
        note: could also be replaced with something like step() to just increment the epoch by one ...
        """
        
        self._current_epoch = epoch

class TestNet(nn.Module):
    def __init__(self, init_mode: str="none"):
        super().__init__()
        
        SZ = 25
        self.linear_1x1 = nn.Linear( 1, SZ)
        self.linear_1x2 = nn.Linear(SZ, SZ)
        self.linear_1x3 = nn.Linear(SZ, SZ)
        self.linear_1x4 = nn.Linear(SZ,  1)
        
        self.activation = nn.ReLU()
        
        
        self._initialize_weights(mode=init_mode)
        return
        
    def _initialize_weights(self, mode: str):
        if mode == "none":
            return
        if mode not in ["xavier", "kaiming", "orthogonal"]:
            raise ValueError(f"please choose a valid method from [xavier, kaiming, orthogonal]! (got {mode})")
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if mode == "xavier":
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
                if mode == "kaiming":
                    nn.init.kaiming_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
                if mode == "orthogonal":
                    nn.init.orthogonal_(m.weight)
                    nn.init.zeros_(m.bias)
        return
    
    def forward(self, sample: dict):
        
        x = sample["input"][:, None].float()
        
        x = self.activation(self.linear_1x1(x))
        x = self.activation(self.linear_1x2(x))
        x = self.activation(self.linear_1x3(x))
        x = self.linear_1x4(x)
        
        return x
    
    
### Functions ----------------------------------------------------------------------------------------------------------
def criterion(sample: dict, pred: torch.Tensor):
    return torch.abs(pred - sample["label"][:, None])

def do_loss(criterion: Callable, sample: dict, pred: torch.Tensor):
    loss        = criterion(sample, pred)
    loss_avg    = torch.mean(loss)
    loss_std_hi = torch.std(loss[loss > loss_avg], unbiased=False)
    loss_std_lo = torch.std(loss[loss < loss_avg], unbiased=False)
    
    return loss_avg, loss_std_hi, loss_std_lo

def do_validation(criterion: Callable, network: nn.Module, validation_dataloader: DataLoader, device: str):
    with torch.no_grad():
        loss = None
        for sample in validation_dataloader:
            sample = {key: value.to(device) for key, value in sample.items()}
            pred   = network(sample)
            
            if loss is None:
                loss = criterion(sample, pred)
            else:
                loss = torch.cat([loss, criterion(sample, pred)])
                
        loss_avg    = torch.mean(loss)
        loss_std_hi = torch.std(loss[loss > loss_avg], unbiased=False)
        loss_std_lo = torch.std(loss[loss < loss_avg], unbiased=False)
        
    return  loss_avg, loss_std_hi, loss_std_lo
    
# plotter stuff  
def calc_network_info(net: nn.Module, opt: torch.optim.Optimizer):
    """
    calculate the effective update rate by getting the effective LR for each param and then multiplying this LR with the gradient for each parameter.
    """
    
    n_params    = 0
    grads       = []
    eff_LRs     = []
    grad_norm   = 0
    eff_LR_norm = 0
    eff_udrate  = 0
    
    # grab all gradient tensors
    for p in net.parameters():
        grads.append(p.grad)
        n_params += torch.numel(p)
    
    # grab all effective LR tensors
    beta1, beta2 = opt.state_dict()["param_groups"][0]["betas"]
    LR           = opt.state_dict()["param_groups"][0]["lr"]
    epsilon      = opt.state_dict()["param_groups"][0].get("eps", 10**-8)
    for key, value in opt.state_dict()["state"].items():
        step    = value["step"]
        m1      = value["exp_avg"]
        m2      = value["exp_avg_sq"]
        m1_corr = m1 / (1 - beta1**step)
        m2_corr = m2 / (1 - beta2**step)
        eff_LR  = (LR * m1_corr) / (torch.sqrt(m2_corr) + epsilon)
        eff_LRs.append(eff_LR)
    
    # calculate effective update rate, grad norm and effective LR
    for grad, eff_LR in zip(grads, eff_LRs): # TODO: check if grads and eff_LRs match up!
        grad_norm   += torch.sum(grad**2)
        eff_LR_norm += torch.sum(eff_LR**2)
        eff_udrate  += torch.sum((grad * eff_LR)**2)

    return grad_norm**0.5, eff_LR_norm**0.5, eff_udrate**0.5

def calc_network_weightnorm(net: nn.Module):
    weight_norm = 0
    for p in net.parameters():
        weight_norm += torch.sum(p**2)
        
    return weight_norm**0.5


### Work ---------------------------------------------------------------------------------------------------------------
DEVICE             = "cuda:0"

# load dataset and split
ds_full            = MyDataset(data)
ds_len             = len(ds_full)
split              = 0.80
ds_train, ds_valid = random_split(ds_full, [round(split * ds_len), ds_len - round(split * ds_len)])

# setup dataloaders and batch schedule
bs_schedule        = [(32, 2), (64, 4), (512, 8), (1028, 8), (2048, 8)]
sampler_train      = ScheduledBatchSampler(ds_train, bs_schedule, shuffle=True, drop_last=True)
loader_train       = DataLoader(ds_train, batch_sampler=sampler_train)
loader_valid       = DataLoader(ds_valid, batch_size=1000)
n_epochs           = sampler_train.get_total_epochs()
n_samples          = sampler_train.get_total_samples()
SAMPLES            = next(iter(loader_valid))

# initialize model, weights and optimizer
model              = TestNet(init_mode="none").to(DEVICE)
optim              = torch.optim.Adam(model.parameters(), lr=10**-3, betas=(0.9, 0.999), weight_decay=0)

# plotter startup app and storage stuff
data_store = {
    "total_samples": n_samples,
    
    "train_loss_X": [], "train_loss_Y": [], 
    "valid_loss_X": [], "valid_loss_Y": [],
    "valid_loss_best_Y": [],
    
    "weight_norm_X": [], "weight_norm_Y": [],
    "grad_norm_X": [], "grad_norm_Y": [],
    
    "batch_size_X": [], "batch_size_Y": [],
    "momentum_X": [], "momentum_Y": [],
    
    "udrate_X": [], "udrate_Y": [],
    
    "n_epochs": 10, "currentepoch": 4,
    "ll_x": np.linspace(-1, 1, 16), "ll_y": np.linspace(-1, 1, 16), "ll_Z": np.random.rand(10, 16, 16) * 10,
    }
modelss_traj = []

app = create_app(data_store)
run_app(app)
sample_counter = 0

# Train Loop -------------------------------------------------------------------
val_avg, val_std_hi, val_std_lo = do_validation(criterion, model, loader_valid, DEVICE)
data_store["valid_loss_X"].append(sample_counter)
data_store["valid_loss_Y"].append(val_avg.detach().cpu().numpy())
data_store["valid_loss_best_Y"] = min(data_store["valid_loss_Y"])

# weight norm for plotting
weight_norm = calc_network_weightnorm(model)

data_store["weight_norm_X"].append(sample_counter)
data_store["weight_norm_Y"].append(weight_norm.detach().cpu().numpy())

# model checkpoint for PCA
modelss_traj.append({k: v.clone().detach().cpu() for k, v in model.state_dict().items()})

for E in range(n_epochs):
    
    model.train()
    sampler_train.set_epoch(E)
    
    for sample in loader_train:
        BS = sample["input"].shape[0]
        
        sample = {key: value.to(DEVICE) for key, value in sample.items()}
        optim.zero_grad()
        pred =  model(sample)
        loss_avg, loss_std_hi, loss_std_lo = do_loss(criterion, sample, pred)
        loss_avg.backward()
        optim.step()
        
        sample_counter += BS
        
        # output stuff for plotting ----------
        
        data_store["batch_size_X"].append(sample_counter)
        data_store["batch_size_Y"].append(BS)        
        
        data_store["train_loss_X"].append(sample_counter)
        data_store["train_loss_Y"].append(loss_avg.detach().cpu().numpy())
            
        grad_norm, eff_LR_norm, eff_udrate = calc_network_info(model, optim)    
        
        data_store["grad_norm_X"].append(sample_counter)
        data_store["grad_norm_Y"].append(grad_norm.detach().cpu().numpy())

        data_store["momentum_X"].append(sample_counter)
        data_store["momentum_Y"].append(eff_LR_norm.detach().cpu().numpy())

        data_store["udrate_X"].append(sample_counter)
        data_store["udrate_Y"].append(eff_udrate.detach().cpu().numpy()) 
        
        # time.sleep(0.01)
    
    # validation (for plotting)    
    val_avg, val_std_hi, val_std_lo = do_validation(criterion, model, loader_valid, DEVICE)
    
    data_store["valid_loss_X"].append(sample_counter)
    data_store["valid_loss_Y"].append(val_avg.detach().cpu().numpy())
    data_store["valid_loss_best_Y"] = min(data_store["valid_loss_Y"])
    
    # weight norm for plotting
    weight_norm = calc_network_weightnorm(model)
    
    data_store["weight_norm_X"].append(sample_counter)
    data_store["weight_norm_Y"].append(weight_norm.detach().cpu().numpy())
    
    # model checkpoint for PCA
    modelss_traj.append({k: v.clone().detach().cpu() for k, v in model.state_dict().items()})
    
    
    torch.cuda.empty_cache()

Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>

In [30]:
# take the saved state dicts and flatten them to [n_chekpoints, n_params]
modelss_traj_flat = []
for statedict in modelss_traj:
    flat = torch.cat([p.view(-1) for p in statedict.values()], dim=0)
    modelss_traj_flat.append(flat.numpy()) 
modelss_traj_flat = np.array(modelss_traj_flat)

# center the trajectory
modelss_traj_centered = modelss_traj_flat - np.mean(modelss_traj_flat, axis=0)

# perform PCA {U}
U, S, VT = np.linalg.svd(modelss_traj_centered, full_matrices=False)
dir1 = VT[0, :]
dir2 = VT[1, :]
evs  = S

# normalize PCA directions with model param norm
param_norm = torch.norm(torch.cat([p.view(-1) for p in modelss_traj[-1].values()])).item()
dir1 = dir1 * param_norm
dir2 = dir2 * param_norm

# reshape dirs
DIR1, DIR2 = {}, {}
idx = 0
for key, value in modelss_traj[-1].items():
    shape     = value.shape
    size      = value.numel()
    DIR1[key] = torch.tensor(dir1[idx:idx+size]).view(shape)
    DIR2[key] = torch.tensor(dir2[idx:idx+size]).view(shape)
    idx += size


fig = go.Figure(go.Scatter(
    mode = "markers",
    y = S,
))

In [23]:
# generate the loss landscape (Full History)
N_STATES = len(modelss_traj)
SAMPLES = {key: value.to(DEVICE) for key, value in SAMPLES.items()}

N    = 16
α, β = np.array([1, 1]) * 0.5
x, y = torch.linspace(-α, α, N), torch.linspace(-β, β, N)
X, Y = torch.meshgrid(x, y, indexing="ij")
Z    = torch.zeros(N_STATES, N, N)

for n in tqdm(range(N_STATES), ncols=120):
    EVAL_STATE = modelss_traj[n]
    for i in range(N):
        for j in range(N):
            tempstate = copy.deepcopy(EVAL_STATE)
            for key, value in tempstate.items():
                tempstate[key] = tempstate[key] + X[i, j] * DIR1[key] + Y[i, j] * DIR2[key]

            model.load_state_dict(tempstate)
            
            pred     = model(SAMPLES)
            loss     = criterion(SAMPLES, pred)
            loss_avg = torch.mean(loss)
            Z[n, i, j] = loss_avg.item()

100%|███████████████████████████████████████████████████████████████████████████████████| 31/31 [00:59<00:00,  1.93s/it]


In [27]:

def create_heatmap(x, y, Z, i):
    trace = go.Heatmap(
        x             = x,
        y             = y,
        z             = torch.log(Z[i, :, :] + 1e-6),
        xgap          = 4,
        ygap          = 4,
        showscale     = False,
        customdata    = Z[i, :, :],
        hovertemplate = "loss: %{customdata:.3f}<extra></extra>",
        visible       = False,
        colorscale    = "Balance",
    )
    return trace

fig  = go.Figure()
init = 5

fac_1     = 1.1
fac_2     = 1.2
x_range_1 = [fac_1 * x.min(), fac_1 * x.max()]
y_range_1 = [fac_1 * y.min(), fac_1 * y.max()]
x_range_2 = [fac_2 * x.min(), fac_2 * x.max()]
y_range_2 = [fac_2 * y.min(), fac_2 * y.max()]

for i in range(N_STATES):
    fig.add_trace(create_heatmap(x, y, Z, i))
    
fig.data[init].visible = True

steps = []
for i in range(N_STATES):
    step = dict(
        method = "update",
        args   = [
            dict(visible = [False] * N_STATES + [True, True]),
        ],
    )
    step["args"][0]["visible"][i] = True
    steps.append(step)
    
sliders = [dict(
    active = init,
    currentvalue = dict(prefix = "nr: "),
    steps = steps,
    x = 0.1,
    y = -0.2,
    len = 0.8,
    xanchor = "left",
    yanchor = "top",
    pad = dict(t = 0, b = 0, l = 0, r = 0),
    font = dict(size = 10, color = "white", family = "JetBrains Mono"),
)]

fig.add_trace(go.Scatter( # axis arrows
    mode       = "markers",
    x          = [x_range_1[0], x_range_2[1]],
    y          = [y_range_2[1], y_range_1[0]],
    marker     = dict(
        size       = 8, 
        symbol     = ["arrow-up", "arrow-right"], 
        color      = "white",
        line_width = [1, 1],
        line_color = "white",
        ),
    hoverinfo  = "skip",
    showlegend = False,
))

fig.add_trace(go.Scatter( # center crosshair
    x          = [0],
    y          = [0],
    marker     = dict(
        size       = 16, 
        symbol     = "cross-thin", 
        line_width = 1,
        line_color = "white"
        ),
    hoverinfo  = "skip",
    showlegend = False
))

fig.update_layout(
    sliders       = sliders,
    
    width         = 500,
    height        = 500,
    margin        = dict(l=20, r=20, t=20, b=20),
    margin_pad    = 10,
    plot_bgcolor  = "rgba(0, 0, 0)",
    paper_bgcolor = "rgba(0, 0, 0)",
    hoverlabel    = dict(
        font        = dict(
            family = "JetBrains Mono", 
            size   = 12, 
            color  = "white"
            ),
        bgcolor     = "black",
        bordercolor = "white",
        ),
    shapes        = [
        dict( # horizontal axis
            type = "line",
            x0   = x_range_2[0], 
            y0   = y_range_1[0],
            x1   = x_range_2[1], 
            y1   = y_range_1[0],
            line = dict(
                color = "white", 
                width = 1
                )
        ),
        dict( # vertical axis
            type = "line",
            x0   = x_range_1[0], 
            y0   = y_range_2[0],
            x1   = x_range_1[0], 
            y1   = y_range_2[1],
            line = dict(
                color = "white", 
                width = 1
                )
        ),
        
    ],
)

fig.update_xaxes(
    title          = "first parameter space principal axis",
    title_font     = dict(
        color  = "white",
        family = "JetBrains Mono",
        size   = 14,
        ),
    title_standoff = 0,
    showgrid       = False,
    zeroline       = False,
    showticklabels = False,
    range          = x_range_2,
    fixedrange     = True,
)

fig.update_yaxes(
    title          = "second parameter space principal axis",
    title_font     = dict(
        color  = "white",
        family = "JetBrains Mono",
        size   = 14,
        ),
    title_standoff = 0,
    scaleanchor    = "x",
    showgrid       = False,
    zeroline       = False,
    showticklabels = False,
    range          = y_range_2,
    fixedrange     = True,
)

fig.show()

In [28]:
# One basic figure (heatmap with gaps) Only One frame!
fac_1 = 1.1
fac_2 = 1.2
x_range_1 = [fac_1 * x.min(), fac_1 * x.max()]
y_range_1 = [fac_1 * y.min(), fac_1 * y.max()]
x_range_2 = [fac_2 * x.min(), fac_2 * x.max()]
y_range_2 = [fac_2 * y.min(), fac_2 * y.max()]

fig = go.Figure()

fig.add_trace(go.Heatmap( # main heatmap 
    x             = x,
    y             = y,
    z             = torch.log(Z[i, :, :] + 1e-6),
    xgap          = 3,
    ygap          = 3,
    showscale     = False,
    customdata    = Z[i, :, :],
    hovertemplate ="loss: %{customdata:.3f}<extra></extra>",
))

fig.add_trace(go.Scatter( # axis arrows
    mode       = "markers",
    x          = [x_range_1[0], x_range_2[1]],
    y          = [y_range_2[1], y_range_1[0]],
    marker     = dict(
        size       = 8, 
        symbol     = ["arrow-up", "arrow-right"], 
        color      = "white",
        line_width = [1, 1],
        line_color = "white",
        ),
    hoverinfo  = "skip",
    showlegend = False,
))

fig.add_trace(go.Scatter( # center crosshair
    x          = [0],
    y          = [0],
    marker     = dict(
        size       = 16, 
        symbol     = "cross-thin", 
        line_width = 1,
        line_color = "white"
        ),
    hoverinfo  = "skip",
    showlegend = False
))

fig.update_layout(
    width         = 400,
    height        = 400,
    margin        = dict(l=20, r=20, t=20, b=20),
    margin_pad    = 10,
    plot_bgcolor  = "rgba(0, 255, 0, 0.)",
    paper_bgcolor = "rgba(0, 255, 0, 0.)",
    hoverlabel    = dict(
        font        = dict(
            family = "JetBrains Mono", 
            size   = 12, 
            color  = "white"
            ),
        bgcolor     = "black",
        bordercolor = "white",
        ),
    shapes        = [
        dict( # horizontal axis
            type = "line",
            x0   = x_range_2[0], 
            y0   = y_range_1[0],
            x1   = x_range_2[1], 
            y1   = y_range_1[0],
            line = dict(
                color = "white", 
                width = 1
                )
        ),
        dict( # vertical axis
            type = "line",
            x0   = x_range_1[0], 
            y0   = y_range_2[0],
            x1   = x_range_1[0], 
            y1   = y_range_2[1],
            line = dict(
                color = "white", 
                width = 1
                )
        ),
        
    ],
)

fig.update_xaxes(
    title          = "first parameter space principal axis",
    title_font     = dict(
        color  = "white",
        family = "JetBrains Mono",
        size   = 14,
        ),
    title_standoff = 0,
    showgrid       = False,
    zeroline       = False,
    showticklabels = False,
    range          = x_range_2,
    fixedrange     = True,
)

fig.update_yaxes(
    title          = "second parameter space principal axis",
    title_font     = dict(
        color  = "white",
        family = "JetBrains Mono",
        size   = 14,
        ),
    title_standoff = 0,
    scaleanchor    = "x",
    showgrid       = False,
    zeroline       = False,
    showticklabels = False,
    range          = y_range_2,
    fixedrange     = True,
)

